# Feature Engineering
Objective: To transform alphanumeric data into mathematical matrices.
- Feature Creation:
    - New feature: Punctuation_Ratio = Punct_Count / Message_Length
    - Remove the two old columns (Punct_Count, Message_Length) to prevent multi-line errors and free up RAM
- Data Split (80/20)
- Feature Extraction:
    - Use the TfidfVectorizer algorithm to transform the Text column into a vocabulary weight matrix.
- Feature Transformation:
    - Apply a logarithmic transformation to the Punctuation_Ratio column to reduce right-bias, handling emails with excessively high punctuation ratios.
    - Apply MinMaxScaler to the column that has been Log Transformed to bring the coordinate system to [0, 1] to synchronize with the TF-IDF matrix.
    (only fit on Train, transform on Test)
- Merge the TF-IDF matrix and the scaled Ratio column into a complete set of X_train_final and X_test_final, ready to load into the model.
- Dimensionality Reduction/Feature Selection (optional)

In [1]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, chi2
from scipy.sparse import hstack, csr_matrix

In [2]:
data = pd.read_csv('./data/preprocessed/enron_spam_data_preprocessed.csv')

In [3]:
pd.set_option('display.max_colwidth', None)
data.head(5)

,Cleaned_Message,Message_Length,Punct_Count,Label
0,calpine daily nomination calpine daily nomination numtoken doc,8,2,0
1,issue fyi see note already stella forward stella morris numtoken numtoken numtoken numtoken numtoken sherlyn schumack numtoken numtoken numtoken numtoken numtoken stella morris howard camp issue stella already take care yesterday thanks howard camp numtoken numtoken numtoken numtoken numtoken stella morris sherlyn schumack howard camp stacey neuweiler daren farmer issue stella work stacey daren resolve forward howard camp numtoken numtoken numtoken numtoken numtoken sherlyn schumack numtoken numtoken numtoken numtoken numtoken howard camp issue create accounting arrangement purchase unocal meter numtoken deal track numtoken numtoken volume deal numtoken expire numtoken numtoken,344,169,0
2,meter numtoken nov allocation fyi forward lauri allen numtoken numtoken numtoken numtoken numtoken kimberly vaughn numtoken numtoken numtoken numtoken numtoken lauri allen mary smith meter numtoken nov allocation lauri put strangas get contract daren forward kimberly vaughn numtoken numtoken numtoken numtoken numtoken lauri allen numtoken numtoken numtoken numtoken numtoken kimberly vaughn anita luong howard camp mary smith meter numtoken nov allocation kim anita volume numtoken show allocate reliant numtoken contract november nomination reliant point november therefore volume allocate contract make sure volume move reliant contract prior november close thanks,318,152,0
3,mcmullen numtoken numtoken jackie since inlet numtoken river plant shut numtoken numtoken numtoken last day flow meter mcmullen divert meter hpl buy residue teco vastar vintage tejones swift still see active deal meter numtoken path manager teco vastar vintage tejones swift also see schedule pop meter numtoken numtoken advice need resolve soon possible settlement send payment thanks,119,21,0
4,meter numtoken jan numtoken george need follow jan numtoken zero numtoken numtoken numtoken numtoken receipt package numtoken allocate flow numtoken numtoken numtoken numtoken numtoken deliv package numtoken jan numtoken zero numtoken numtoken numtoken numtoken receipt package numtoken zero numtoken numtoken numtoken numtoken deliv package numtoken buyback incorrectly nominate transport contract numtoken receipt let know,90,16,0


In [4]:
# Đảm bảo không có dòng text nào bị NaN (đôi khi hàm clean_text xóa sạch mọi thứ của 1 email ngắn)
data = data.dropna(subset=['Cleaned_Message']).reset_index(drop=True)

In [5]:
# Tránh lỗi chia cho 0 nếu có email length = 0
data['Message_Length'] = data['Message_Length'].replace(0, 1)

# Tạo Interaction Feature: Tỷ lệ dấu câu
data['Punctuation_Ratio'] = data['Punct_Count'] / data['Message_Length']

In [6]:
# Drop cột cũ để triệt tiêu đa cộng tuyến và dọn RAM
data = data.drop(columns=['Punct_Count', 'Message_Length'])

In [7]:
data.head(5)

,Cleaned_Message,Label,Punctuation_Ratio
0,calpine daily nomination calpine daily nomination numtoken doc,0,0.250000
1,issue fyi see note already stella forward stella morris numtoken numtoken numtoken numtoken numtoken sherlyn schumack numtoken numtoken numtoken numtoken numtoken stella morris howard camp issue stella already take care yesterday thanks howard camp numtoken numtoken numtoken numtoken numtoken stella morris sherlyn schumack howard camp stacey neuweiler daren farmer issue stella work stacey daren resolve forward howard camp numtoken numtoken numtoken numtoken numtoken sherlyn schumack numtoken numtoken numtoken numtoken numtoken howard camp issue create accounting arrangement purchase unocal meter numtoken deal track numtoken numtoken volume deal numtoken expire numtoken numtoken,0,0.491279
2,meter numtoken nov allocation fyi forward lauri allen numtoken numtoken numtoken numtoken numtoken kimberly vaughn numtoken numtoken numtoken numtoken numtoken lauri allen mary smith meter numtoken nov allocation lauri put strangas get contract daren forward kimberly vaughn numtoken numtoken numtoken numtoken numtoken lauri allen numtoken numtoken numtoken numtoken numtoken kimberly vaughn anita luong howard camp mary smith meter numtoken nov allocation kim anita volume numtoken show allocate reliant numtoken contract november nomination reliant point november therefore volume allocate contract make sure volume move reliant contract prior november close thanks,0,0.477987
3,mcmullen numtoken numtoken jackie since inlet numtoken river plant shut numtoken numtoken numtoken last day flow meter mcmullen divert meter hpl buy residue teco vastar vintage tejones swift still see active deal meter numtoken path manager teco vastar vintage tejones swift also see schedule pop meter numtoken numtoken advice need resolve soon possible settlement send payment thanks,0,0.176471
4,meter numtoken jan numtoken george need follow jan numtoken zero numtoken numtoken numtoken numtoken receipt package numtoken allocate flow numtoken numtoken numtoken numtoken numtoken deliv package numtoken jan numtoken zero numtoken numtoken numtoken numtoken receipt package numtoken zero numtoken numtoken numtoken numtoken deliv package numtoken buyback incorrectly nominate transport contract numtoken receipt let know,0,0.177778


In [8]:
X = data[['Cleaned_Message', 'Punctuation_Ratio']]
y = data['Label']

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [10]:
X_train.shape[0]

21906

In [11]:
X_test.shape[0]

5477

In [12]:
# Giới hạn 5000 features tốt nhất, xét cả từ đơn và cụm 2 từ (Bigrams)
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

# Fit và Transform trên Train, chỉ Transform trên Test
X_train_tfidf = tfidf.fit_transform(X_train['Cleaned_Message'])
X_test_tfidf = tfidf.transform(X_test['Cleaned_Message'])

In [13]:
# Feature Transformation
X_train_ratio_log = np.log1p(X_train['Punctuation_Ratio'].values.reshape(-1, 1))
X_test_ratio_log = np.log1p(X_test['Punctuation_Ratio'].values.reshape(-1, 1))

In [14]:
# Min-Max Scaling (Đưa về [0, 1] cho đồng bộ với TF-IDF)
scaler = MinMaxScaler()
X_train_ratio_scaled = scaler.fit_transform(X_train_ratio_log)
X_test_ratio_scaled = scaler.transform(X_test_ratio_log)

In [15]:
# Ép kiểu về Sparse Matrix để merge
X_train_ratio_sparse = csr_matrix(X_train_ratio_scaled)
X_test_ratio_sparse = csr_matrix(X_test_ratio_scaled)

In [16]:
X_train_final = hstack([X_train_tfidf, X_train_ratio_sparse])
X_test_final = hstack([X_test_tfidf, X_test_ratio_sparse])

In [17]:
X_train_final.shape

(21906, 5001)

In [18]:
X_test_final.shape

(5477, 5001)

In [19]:
os.makedirs('./data/ready_for_train', exist_ok=True)

In [20]:
joblib.dump(X_train_final, './data/ready_for_train/X_train_final.pkl')
joblib.dump(X_test_final, './data/ready_for_train/X_test_final.pkl')
joblib.dump(y_train, './data/ready_for_train/y_train.pkl')
joblib.dump(y_test, './data/ready_for_train/y_test.pkl')

['./data/ready_for_train/y_test.pkl']

In [21]:
joblib.dump(tfidf, './data/ready_for_train/tfidf_vectorizer.pkl')
joblib.dump(scaler, './data/ready_for_train/minmax_scaler.pkl')

['./data/ready_for_train/minmax_scaler.pkl']